# Parameter values, methods and settings

<div class="alert alert-info">

**Summary**

This lesson binds the consumption stage of the portfolio–consumption model to its parameter values, numerical methods and numerical settings, and reads after each binding what changes on the card of the decision builder and on the kernel of its aggregation. The object inspected is the stage at its four versions. The lesson evaluates one kernel at one point and solves nothing.

</div>

<div class="alert alert-success">

**Relationship with other documents**

**Builds on** — [The stage as an object](01-the-stage.ipynb) and [Value functions and the backward builders](02-value-functions.ipynb): the stage and its cards.

**Formal account** — [Methodization](../../docs/Bellman-Sym/04-methodization.md): where a method may stand in a stage, the options it reads and the settings that give them values.

**Used by** — [Periods and trellises](05-periods-and-trellises.ipynb): the three bindings of every stage of a trellis.

**See also** — [The portfolio–consumption application](../../applications/cons-port/README.md): the files read here.

</div>

<div class="alert">

**Key terminology**

| Term | Meaning |
|---|---|
| [[symbolic_stage\|symbolic]], [[specified_stage\|specified]] | Said of a stage with no binding, and with all three. |
| place | Where in a stage a method may stand, such as a builder, a line of a block, an operation or a declared function. |
| [[handler]] | The function of an application's tool that carries a method out. |

</div>

The [[stage]] is the consumption stage of the first lesson: the household arrives with resources $\mathrm m$, chooses consumption $\mathrm c$ in $(0, \mathrm m_d]$ and carries savings $\mathrm a = \mathrm m_d - \mathrm c$ forward, with utility $u(\mathrm c) = \mathrm c^{1-\rho}/(1-\rho)$ and relative risk aversion $\rho$.

The stage file declares $\rho$ but gives it no value, and it names no grid and no method of solution. Three files supply these, and three verbs of the stage bind them, in this order: `calibrate` binds the values of the [[parameters]], `methodize` attaches numerical methods to places of the stage, and `configure` gives values to the [[settings]] the methods read.

The lesson reads six files and writes none:

- `applications/cons-port/declaration/stages/cons/cons.bl`
- `applications/cons-port/declaration/calibration/portfolio.yml`
- `applications/cons-port/declaration/methods/cons.yml`
- `applications/cons-port/declaration/methods/cons_egm.bl`
- `applications/cons-port/declaration/methods/cons_vfi.yml`
- `applications/cons-port/declaration/settings/cons.yml`

The first code cell imports the collection's set-up module, checkout, which the first lesson describes, the three names of the package the lesson calls, and the package's reader of a calibration file, load_calibration of the module bellman.stage.sources.

In [1]:
import checkout
from bellman.stage import elaborate_stage, load_stage, method_types
from bellman.stage.sources import load_calibration

## The symbolic stage

The next cell constructs the stage from its file, as the first lesson does, and names it `cons`.

In [2]:
cons = elaborate_stage(load_stage("applications/cons-port/declaration/stages/cons/cons.bl"))

`par` answers a table of the parameters the file declares, each with the set its declaration names, its value and, under `describes`, the description a declaration may write after `::`, none here. The file's line `ρ : [1, inf)` declares $\rho$ in $[1, \infty)$, and no line gives it a value.

In [3]:
cons.par

name  set       value    describes
ρ     (1, inf)  unbound

A declaration writes its description as a quoted entry in the bracketed list a line may carry after `::`, the list in which the line defining utility writes its identifier, `[id=util]`. The next cell constructs the stage again from a copy of the file's text in which the declaration of $\rho$ reads `ρ : [1, inf)  :: ["relative risk aversion"]`, and names it `cons_described`; the file itself is unchanged. The cell gives `elaborate_stage` the copy in the form `load_stage` returns, a mapping that holds under `stage` the text and a name for its origin, and holds no calibration, settings or methods.

In [4]:
cons_described = elaborate_stage({
    "stage": {
        "text": open("applications/cons-port/declaration/stages/cons/cons.bl").read().replace(
            "ρ : (1, inf)", 'ρ : (1, inf)  :: ["relative risk aversion"]'),
        "origin": "cons-described.bl",
    },
    "calibration": {},
    "settings": {},
    "methods": {},
})

Its table of parameters prints the description under `describes`, and `cons_described.par["ρ"].describes` answers the same text for $\rho$ alone.

In [5]:
cons_described.par

name  set       value    describes
ρ     (1, inf)  unbound  relative risk aversion

`elaborate_stage` binds the parameter values a record holds under `calibration`, as the verb `calibrate` of the section on parameter values below binds them, and answers the symbolic stage, as above, for a record that holds none: the next cell gives it the same text with `applications/cons-port/declaration/calibration/portfolio.yml` read into `calibration`, and its table of parameters holds the value of $\rho$ beside the description.

In [6]:
elaborate_stage({
    "stage": {
        "text": open("applications/cons-port/declaration/stages/cons/cons.bl").read().replace(
            "ρ : (1, inf)", 'ρ : (1, inf)  :: ["relative risk aversion"]'),
        "origin": "cons-described.bl",
    },
    "calibration": load_calibration("applications/cons-port/declaration/calibration/portfolio.yml"),
    "settings": {},
    "methods": {},
}).par

name  set       value  describes
ρ     (1, inf)  6.0    relative risk aversion

A parameter is named in the definitions that use it. The next cell asks for the card of utility, the printed statement of that definition, by its key. Under `Conditions` the card records one condition the construction does not verify: $1 - \rho$ is not zero, since the formula divides by it.

In [7]:
cons.definitions.denote("cons:function:𝑢")

STAGE cons / FUNCTION 𝑢
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Defines
  𝑢  : ℝ₊₊ → (−∞, 0),  c ↦ c^(1 − ρ)/(1 − ρ)
──────────────────────────────────────────────────────────────────────────────
Source  cons.bl
  5  𝑢(c : ℝ₊₊) := c^(1-ρ)/(1-ρ)

Definitions referenced
  ρ     cons:parameter:ρ
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

`bindings` shows the three bindings of the version, each absent here and printed beside the verb that supplies it.

In [8]:
cons.bindings

calibration   none   calibrate(stage, source)
methods       none   methodize(stage, source)
settings      none   configure(stage, source)

## Parameter values

A calibration file gives parameter values as one mapping. This one serves the four stages of the model, so it holds more names than the consumption stage declares.

In [9]:
print(open("applications/cons-port/declaration/calibration/portfolio.yml").read())

# The parameter values of the portfolio-consumption model, those the fixed reference was computed with
# (reference/legacy/syntax/calibration.yaml). One mapping serves the four stages; each stage binds the parameters it declares.
calibration:
  β: 0.96
  ρ: 6.0
  R: 1.02
  μ_Ψ: 0.028927
  σ_Ψ: 0.143486
  μ_θ: -0.01125
  σ_θ: 0.15
  ρ_ζ: 0.0



[[calibration|Calibration]] binds the values. `calibrate` checks the value of each declared parameter against its set and, at those values, decides each condition that involves only parameters; it refuses a value outside its set or one that makes a condition false. Of the file's names it binds `ρ` alone, and it returns a new version of the stage, which the next cell names `cons_calibrated`.

In [10]:
cons_calibrated = cons.calibrate("applications/cons-port/declaration/calibration/portfolio.yml")

Its table of parameters carries the value. One parameter is read by the name the file declares: `cons_calibrated.par["ρ"]` answers the record of $\rho$ alone, and its `value` is the number bound, 6.0.

In [11]:
cons_calibrated.par


name  set       value  describes
ρ     (1, inf)  6.0

Written as an attribute, the name answers the value alone: the next cell shows the record `cons_calibrated.par["ρ"]` beside `cons_calibrated.par.ρ`, the number 6.0. On the symbolic `cons`, which binds no value, `cons.par.ρ` is refused and the record is read by the key.

In [12]:
cons_calibrated.par["ρ"], cons_calibrated.par.ρ


(ρ = 6.0 : (1, inf), 6.0)

The next cell prints the card of utility on the calibrated version. The formula still names $\rho$, whose value the version holds in its bindings, beside the definition. The condition on $1 - \rho$ no longer stands under `Conditions`: calibration decided it at $\rho = 6.0$, where it holds.

In [13]:
cons_calibrated.definitions.denote("cons:function:𝑢")

STAGE cons / FUNCTION 𝑢
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Defines
  𝑢  : ℝ₊₊ → (−∞, 0),  c ↦ c^(1 − ρ)/(1 − ρ)
──────────────────────────────────────────────────────────────────────────────
Source  cons.bl
  5  𝑢(c : ℝ₊₊) := c^(1-ρ)/(1-ρ)

Definitions referenced
  ρ     cons:parameter:ρ
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The verb returned a new version and left the one it was called on unchanged, as their short forms show.

In [14]:
cons, cons_calibrated

(ElaboratedStage('cons', calibrated), ElaboratedStage('cons', calibrated))

## The kernel at the values

A **kernel** is the mathematical data an operation uses: a law of motion for a pullback or a push-forward, a value aggregator for an aggregation, a probability distribution for an expectation. The decision builder's aggregation applies $G(v, \mathrm m_d, \mathrm c) = \big(u(\mathrm c) + v,\ u'(\mathrm c)\big)$, which gives the value $V$ and the marginal value $dV$ at the decision when $v$ is the pulled-back continuation value; $u'$ is marginal utility, `𝑑𝑢` in the file. On the calibrated version the kernel is the line of the symbolic version, with $\rho$ still a name inside $u$.

In [15]:
cons_calibrated.dcsn.builder.bwd["Aggregation"].kernel()

𝐺 of cntn_to_dcsn: 𝐺(V≻, m, c) = 𝑢(c) + V≻
output V; unused m

`compile_kernel()` turns the kernel into a Python function whose arguments are the kernel's own, in the printed order, followed by each parameter the kernel reads, here $\rho$. The answer of `kernel()` is a record, displayed as the rule above. Under `parameter_values` it holds the calibration's values, here `(6.0,)`, which a caller passes to the function at each call; under `python` it holds the function's source text, which the next cell prints. A symbolic version compiles too, and its record holds `None` under `parameter_values`.

In [16]:
print(cons_calibrated.dcsn.builder.bwd["Aggregation"].kernel().python)

# bellman kernel of cons, cntn_to_dcsn:Aggregation; cons.bl sha256 34cf564877c0d7a4ecd5e15dcd55ed776ffe7f649a1cb674d378753b33cdea78; bellman 0.1.6
# 𝐺 of cntn_to_dcsn: 𝐺(V≻, m, c) = 𝑢(c) + V≻
# output V; unused m
# parameters: ρ ∈ (1, inf)
# names: V_cntn is V≻; u is 𝑢
import numpy as np

def G_cntn_to_dcsn(V_cntn, m, c, ρ):
    def u(c):
        return c**(1 - ρ)/(1 - ρ)
    V = u(c) + V_cntn
    return V



The next cell evaluates the compiled kernel at continuation value $V_\succ = 4$, resources $\mathrm m_d = 2$ and consumption $\mathrm c = 0.5$.

In [17]:
print(cons_calibrated.dcsn.builder.bwd["Aggregation"].compile_kernel()(4.0, 2.0, 0.5, *cons_calibrated.dcsn.builder.bwd["Aggregation"].kernel().parameter_values))

-2.4000000000000004


The value is $u(0.5) + 4 = 0.5^{-5}/(-5) + 4 = -2.4$, printed as $-2.4000000000000004$ because $-6.4$ has no exact binary representation and the sum keeps its rounding. The kernel has one output: the stage file writes no marginal value, and the methods file of the next section adds the marginal value $u'(c)$ by a line the author writes in a marginal block.

## Methods

A methods file is a list of entries, which [[methodization]] attaches to the stage. In an entry, `on:` names a place of the stage and `method:` the tag of a numerical method; `handler:` names the [[handler]], the function of the application's numerical tool that carries the method out; `settings:` gives the options the method reads, each by the name of a setting. The first entry binds the endogenous-grid method on the policy builder with the block file `cons_egm.bl`, whose method block computes the policy, and names its exogenous grid of savings by the setting `grid`, whose value is the grid's kind, the tag `!Cartesian`, with the grid's order and bounds beside it. The second binds the block file's marginal block, the envelope condition, on the decision builder with `!marginals`. The third represents the value and its stated derivative by the basis `!Linear`, linear interpolation between their values on the grid of resources that its setting `grid` names in the same way, at the decision builder's Evaluation, which the entry names by the derivative, `cntn_to_dcsn.evaluate.marginal`.

In [18]:
print(open("applications/cons-port/declaration/methods/cons.yml").read())

stage: cons
# The consumption stage's methods. The stage file writes the maximization alone; the block file cons_egm.bl,
# beside this file, opens the policy builder with the endogenous-grid method and states the derivative of the
# decision value, the envelope condition. The function egm of the application's tool,
# applications/cons-port/code/cons_port_tool.py, realizes the method on the exogenous grid of savings a that its
# setting grid names: the kind of the grid is the tag on the value of grid, !Cartesian, and its order and bounds, the
# settings n_a, a_min and a_max, stand beside it. The value and its stated derivative are represented at the decision
# builder's Evaluation by the basis !Linear, read between its nodes by linear interpolation, on the grid of
# resources m that the setting grid names in the same way; the entry names that place by the derivative,
# cntn_to_dcsn.evaluate.marginal.
methods:
  - on: policy
    method: !egm
    block: cons_egm.bl
    handler: cons-port:

`places()` lists every address at which a method may stand, with its kind: the three builders; the named line `policy.optimality`, the maximization, of kind `statement`; the operator `argmax` in that line, by its address and by its stage-wide name; the Evaluations that close the decision builder and the policy builder, `cntn_to_dcsn.evaluate.value` for the value function and `policy.evaluate` for the policy function, of kind `operator evaluate`, where a representation is bound; and the declared function $u$ and the correspondence $\Gamma$, of kind `definition`. The `on:` of an entry names one of them, or, with `block:`, a builder that a block of the block file opens.

In [19]:
cons.places()

address                      kind
dcsn_to_cntn                 builder
cntn_to_dcsn                 builder
cntn_to_dcsn.evaluate.value  operator evaluate
policy                       builder
policy.optimality            statement
policy.optimality.argmax     operator argmax
policy.evaluate              operator evaluate
argmax                       operator argmax
𝑢                            definition
Γ                            definition

`methodize` attaches the entries to the calibrated version, following the order of the verbs, and returns a new version, which the next cell names `cons_methodized`.

In [20]:
cons_methodized = cons_calibrated.methodize("applications/cons-port/declaration/methods/cons.yml")

`methods` lists each binding under the place it stands at. `!egm` stands on the builder `policy` itself, beside the block file `cons_egm.bl`: the block `egm` opens the policy builder, and its three lines, the candidate consumption at each saving, the resources at which that saving is chosen and the policy read on those resources, become the builder's operations. `!marginals` stands at the Evaluation that closes the decision value builder, `cntn_to_dcsn:Evaluation`, where the marginal block adds the stated derivative beside the value. The representation `!Linear` stands beside `!marginals` at `cntn_to_dcsn:Evaluation`, since a place holds one binding of each kind.

In [21]:
cons_methodized.methods

policy                    !egm   block cons_egm.bl
cntn_to_dcsn:Evaluation   !marginals   block cons_egm.bl
cntn_to_dcsn:Evaluation   !Linear

The evaluation that closes the opened policy builder reads the consumption rule $c$ from the block's lines: at each level of savings $\mathrm a$, the consumption $c_\succ(\mathrm a)$ the inverse Euler equation allows and the resources $m_{d\succ}(\mathrm a) = \mathrm a + c_\succ(\mathrm a)$ that lead to it. The first line of the card's `Rule` states $c$ on the resources so reached, $m_{d\succ}(\mathrm{Xa})$, the endogenous grid; the second, the declared set of feasible rules, into which the evaluation returns $c$ unchanged. The method stands on the builder and not on this evaluation, so the card has no `Method` heading; the builder's card, `cons_methodized.builder.policy.denote()`, prints it.

In [22]:
cons_methodized.dcsn.builder.policy["Evaluation"].denote()

STAGE cons / POLICY BUILDER policy / Evaluation
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)
       ───────────────────────────
  out  𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Rule
  𝑐(𝑚≻(a)) = 𝑐≻(a)                                                   on 𝑚≻(Xa)
  𝑐 : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)
──────────────────────────────────────────────────────────────────────────────
Definitions referenced
  Xm    cons:set:Xm
  Xa    cons:set:Xa
  Γ     cons:correspondence:Γ
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

The card of the decision builder gains, below `Conditions`, a `Method` heading for each binding its evaluation holds: `!marginals` with the block file, whose marginal block writes the marginal value beside the value, and `!Linear`, the representation, with no handler named. `Options` gives the grid's kind, `!Cartesian`, and beside it the grid's orders, the number of points along each coordinate, and its bounds, by the names of the settings, which have no values yet.

In [23]:
cons_methodized.dcsn.builder.bwd.denote()

STAGE cons / BACKWARD BUILDER cntn → dcsn (cntn_to_dcsn)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ───────────────────────────────────────────────────
  out  𝑉   : Xm → ℝ

At
  𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻(·, 𝑐)
     ─────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻(·, 𝑐))(m) = 𝑉≻(𝑔≻(m, 𝑐(m)))

  2  Aggregation
     in: (1)                            out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉(m) = 𝑢(𝑐(m)) + 𝑉≻(𝑔≻(m, 𝑐(m)))

  3  Evaluation
     in: 𝑉                              out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉 : Xm → ℝ
           D_{m}𝑉(m) = 𝑑𝑢(𝑐(m))

Conditions
  Aggregation  𝑉(m) ∈ ℝ

Method
  !marginals   block cons_egm.bl                       cntn_to_dcsn:Evaluation

Method
  !Linear   handler                                  

`method_types` lists the methods the package knows. The entry of a tag prints as a card that gives, under `Admits`, the kinds of place at which the method may stand and, under `Options`, the options it reads: for `!Linear`, four kinds of place and one required option, `grid`, whose value is the grid's kind written as a tag; the fields that kind admits, `orders` and `bounds` among them, stand beside it in the entry's settings. Of the four kinds, `node` is a line marked `!node`, `statement` a value line, `definition` a declared function, and `operator evaluate` an evaluation: an authored `evaluate(…)`, or the Evaluation that closes a builder, which the methods file names `<block-id>.evaluate`, `<block-id>.evaluate.value` or `<block-id>.evaluate.marginal`.

In [24]:
method_types["!Linear"]

METHOD !Linear
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Admits
  node
  statement
  definition
  operator evaluate

Options
  grid, grid kind (required)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

A method is refused at a place of a kind it does not admit. Gauss–Hermite quadrature computes an expectation; an entry that attaches it to the builder `policy` is refused. The message calls a place a **target** and names the kind of place that admits the method, an expectation operator, of which this stage has none.

In [25]:
cons_calibrated.methodize({"stage": "cons", "methods": [{"on": "policy", "method": "!gauss-hermite"}]})

TargetError: Method target 'policy' in stage 'cons': !gauss-hermite does not admit target kind builder; method-attachment admits operator E; write an on: target of one of these kinds

The second methods file, which the next cell prints, binds no block: it writes a grid search over candidate consumption levels, `!vfi`, on the decision builder `cntn_to_dcsn`, and the representation of the value at the decision builder's Evaluation, `cntn_to_dcsn.evaluate.value`.

In [26]:
print(open("applications/cons-port/declaration/methods/cons_vfi.yml").read())

stage: cons
# The consumption stage's methods under value function iteration, on the stage file
# declaration/stages/cons/cons.bl, which writes the maximization alone: a
# grid search over candidate consumption levels on the backward decision builder, realized by the function
# consumption_search of cons_port_tool.py, and the value represented at the decision builder's Evaluation,
# cntn_to_dcsn.evaluate.value, by the basis !Linear on the grid of resources m that the setting grid names by its
# kind, !Cartesian, with its order and bounds, the settings n_m, m_min and m_max, beside it.
# No block is bound: the search reads no marginal value of the continuation and takes the marginal value of the
# decision as the difference quotient of the values.
methods:
  - on: cntn_to_dcsn
    method: !vfi
    handler: cons-port:consumption_search
  - on: cntn_to_dcsn.evaluate.value
    method: !Linear
    settings: {grid: !Cartesian, orders: {m: n_m}, bounds: {m: [m_min, m_max]}}



`methodize` binds the stage it is called on, and `cons_calibrated` now holds the block, so the next cell binds the second file on a stage of its own, elaborated and calibrated again from the same files.

In [27]:
cons_methodized_vfi = elaborate_stage(load_stage("applications/cons-port/declaration/stages/cons/cons.bl")).calibrate(
    "applications/cons-port/declaration/calibration/portfolio.yml").methodize("applications/cons-port/declaration/methods/cons_vfi.yml")

`!vfi` stays where the entry wrote it, on the decision builder `cntn_to_dcsn`, the one builder at which it may stand: at each level of resources the grid search maximizes over candidate consumption levels the value that builder computes, $u(\mathrm c) + V_\succ(\mathrm m_d - \mathrm c)$. The policy builder keeps the maximization the stage file writes, which no block opens here.

In [28]:
cons_methodized_vfi.methods

cntn_to_dcsn              !vfi
cntn_to_dcsn:Evaluation   !Linear

## Settings

A settings file gives values to the names the methods read: here the grid of resources, 100 points from 0.01 to 10.0, and in its last three names the grid of savings of the endogenous-grid method, as its comment says.

In [29]:
print(open("applications/cons-port/declaration/settings/cons.yml").read())

# The consumption stage's numerical settings: the grid of resources on which the value is represented, the
# default the earlier helper took (ffp_utils.py line 431), and the savings grid of the endogenous-grid method
# (ffp_utils.py lines 713 to 715).
settings:
  n_m: 100
  m_min: 0.01
  m_max: 10.0
  n_a: 800
  a_min: 1.0e-6
  a_max: 100.0



`configure` binds the settings to the methodized version and returns the **specified** version, which the next cell names `cons_specified`. It requires bound methods, whose options name the settings, and it checks each option at the values: an order must be a positive integer, and a bound an ordered pair inside the set the file declares for its coordinate, `Xm` for resources.

In [30]:
cons_specified = cons_methodized.configure("applications/cons-port/declaration/settings/cons.yml")

`settings` lists each setting with its value, the place that reads it, the method that reads it there and, under `grid`, the kind of the grid of which the setting is a field. The first three are read at `cntn_to_dcsn:Evaluation` by `!Linear` as its grid of resources, and the last three at `policy` by `!egm` as its exogenous grid of savings; both grids are of the kind `!Cartesian`. One setting is read by its name in the same way, `cons_specified.settings["n_m"]`, whose `value` is 100.

In [31]:
cons_specified.settings

name   value  read at                  by       grid
n_m    100    cntn_to_dcsn:Evaluation  !Linear  !Cartesian
m_min  0.01   cntn_to_dcsn:Evaluation  !Linear  !Cartesian
m_max  10.0   cntn_to_dcsn:Evaluation  !Linear  !Cartesian
n_a    800    policy                   !egm     !Cartesian
a_min  1e-06  policy                   !egm     !Cartesian
a_max  100.0  policy                   !egm     !Cartesian

A setting's name written as an attribute answers its value alone, as a parameter's does: the next cell shows the record `cons_specified.settings["n_a"]` beside `cons_specified.settings.n_a`, the number 800.

In [32]:
cons_specified.settings["n_a"], cons_specified.settings.n_a

(n_a = 800, read at policy by !egm for its grid !Cartesian, 800)

The next cell prints the decision builder's card on the specified version. The `Options` lines now hold the values, 100 points between 0.01 and 10.0, and no other line has changed.

In [33]:
cons_specified.dcsn.builder.bwd.denote()

STAGE cons / BACKWARD BUILDER cntn → dcsn (cntn_to_dcsn)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Signature
  in   𝑉≻  : Xa → ℝ
       ───────────────────────────────────────────────────
  out  𝑉   : Xm → ℝ

At
  𝑐  : Xm → ℝ₊₊,  𝑐(m) ∈ Γ(m)

Operations
  1  Pullback
     in: 𝑉≻                             out: 𝑉≻ ∘ 𝑔≻(·, 𝑐)
     ─────────────────────────────────────────────────────
     rule: (𝑉≻ ∘ 𝑔≻(·, 𝑐))(m) = 𝑉≻(𝑔≻(m, 𝑐(m)))

  2  Aggregation
     in: (1)                            out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉(m) = 𝑢(𝑐(m)) + 𝑉≻(𝑔≻(m, 𝑐(m)))

  3  Evaluation
     in: 𝑉                              out: 𝑉
     ─────────────────────────────────────────────────────
     rule: 𝑉 : Xm → ℝ
           D_{m}𝑉(m) = 𝑑𝑢(𝑐(m))

Conditions
  Aggregation  𝑉(m) ∈ ℝ

Method
  !marginals   block cons_egm.bl                       cntn_to_dcsn:Evaluation

Method
  !Linear   handler                                  

The aggregation's kernel on the specified version is the line of the section [The kernel at the values](#The-kernel-at-the-values): a method and its settings say how a function is computed and stored, and they change no equation.

In [34]:
cons_specified.dcsn.builder.bwd["Aggregation"].kernel()

𝐺 of cntn_to_dcsn: 𝐺(V≻, m, c) = 𝑢(c) + V≻
output V; unused m

## The four versions

The four short forms, from the symbolic version to the specified one; each binding left the version it was called on unchanged.

In [35]:
cons, cons_calibrated, cons_methodized, cons_specified

(ElaboratedStage('cons', specified),
 ElaboratedStage('cons', specified),
 ElaboratedStage('cons', specified),
 ElaboratedStage('cons', specified))

The next cell shows the specified version's bindings: the value of $\rho$, the three methods by their places, and the six settings.

In [36]:
cons_specified.bindings

calibration   ρ = 6.0
methods       policy                    !egm   block cons_egm.bl
              cntn_to_dcsn:Evaluation   !marginals   block cons_egm.bl
              cntn_to_dcsn:Evaluation   !Linear
settings      n_m     100     cntn_to_dcsn:Evaluation   !Linear
              m_min   0.01    cntn_to_dcsn:Evaluation   !Linear
              m_max   10.0    cntn_to_dcsn:Evaluation   !Linear
              n_a     800     policy   !egm
              a_min   1e-06   policy   !egm
              a_max   100.0   policy   !egm

The package constructs and binds the stage; an application's numerical tool, whose functions the handlers name, solves it.